# RoBERTa Model Training Across All Dataset Variants
## Compute Environment: Google Colab (T4 GPU)

Fine-tune a RoBERTa-base model as a binary entity matching classifier.

Architecture mirrors Ditto:
  - Model  : roberta-base
  - Input  : [CLS] left_entity [SEP] right_entity [SEP]
  - Output : binary classification (0 = non-match, 1 = match)
  - Loss   : cross-entropy


In [ ]:
!unzip -q data.zip

In [ ]:
import sys
from pathlib import Path
import json

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from dotenv import load_dotenv

load_dotenv()

# from metrics import compute_metrics, full_report, confusion

Metrics

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


def compute_metrics(labels: list[int], predictions: list[int]) -> dict:
    """Return precision, recall, and F1 for the positive (match) class."""
    return {
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }


def full_report(labels: list[int], predictions: list[int]) -> str:
    """Full sklearn classification report as a string."""
    return classification_report(labels, predictions, target_names=["non-match", "match"])


def confusion(labels: list[int], predictions: list[int]) -> dict:
    """Return TP, FP, FN, TN counts."""
    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

Data loading

In [ ]:
ROOT = Path().cwd()
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

MODELS = ROOT / "experiments" / "models"
MODELS.mkdir(parents=True, exist_ok=True)

RESULTS = ROOT / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_pairs(path: Path) -> tuple[list[str], list[str], list[int]]:
    """Read a Ditto-format .txt file into (lefts, rights, labels)."""
    lefts, rights, labels = [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 3:
                continue
            lefts.append(parts[0])
            rights.append(parts[1])
            labels.append(int(parts[2]))
    return lefts, rights, labels

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256  # max tokens per pair



def build_hf_dataset(
    path: Path,
    tokenizer,
    max_length: int = MAX_LENGTH,
) -> Dataset:
    lefts, rights, labels = load_pairs(path)

    encodings = tokenizer(
        lefts,
        rights,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )
    encodings["labels"] = labels
    return Dataset.from_dict(encodings)

HuggingFace compute_metrics callback

In [ ]:
def make_compute_metrics_fn():
    def _compute(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1).tolist()
        labels = labels.tolist()
        return compute_metrics(labels, preds)

    return _compute

Make weighted trainer

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer subclass that applies class weights to the cross-entropy loss."""

    def __init__(self, class_weights: torch.Tensor, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_class_weights(labels: list[int]) -> torch.Tensor:
    """Inverse-frequency weights: w_c = n_total / (n_classes * n_c)."""
    counts = [labels.count(0), labels.count(1)]
    n = len(labels)
    weights = [n / (2 * c) if c > 0 else 1.0 for c in counts]
    print(f"  Class weights: non-match={weights[0]:.3f}, match={weights[1]:.3f}")
    return torch.tensor(weights, dtype=torch.float)

Training

In [ ]:
import os
import shutil

def delete_checkpoint_folders(path):
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path) and item.startswith("checkpoint"):
            shutil.rmtree(item_path)
            print(f"Deleted: {item_path}")

In [ ]:
def train(
    dataset: str,
    epochs: int = 10,
    batch_size: int = 8,
    grad_accum: int = 4,
    lr: float = 2e-5,
    seed: int = 42,
    class_weight: str = "none",
    run_name: str = "baseline",
    train_file: Path | None = None,
):
    data_dir = PROCESSED / dataset
    if train_file is None:
        train_file = data_dir / "train.txt"
    valid_file = data_dir / "valid.txt"

    if not train_file.exists():
        sys.exit(
            f"[error] {train_file} not found. Run src/data_prep/preprocess.py first."
        )

    model_out = MODELS / f"{run_name}_{dataset}"
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Training {run_name} on {dataset} ===")
    print(f"  Model        : {MODEL_NAME}")
    print(f"  Train file   : {train_file}")
    print(f"  Epochs       : {epochs}")
    print(f"  Batch size   : {batch_size}  (grad_accum={grad_accum}, effective={batch_size*grad_accum})")
    print(f"  LR           : {lr}")
    print(f"  Class weight : {class_weight}")
    print(f"  Output dir   : {model_out}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    print("  Tokenizing training set...")
    train_ds = build_hf_dataset(train_file, tokenizer)

    eval_ds = None
    if valid_file.exists():
        print("  Tokenizing validation set...")
        eval_ds = build_hf_dataset(valid_file, tokenizer)

    training_args = TrainingArguments(
        output_dir=str(model_out),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch" if eval_ds else "no",
        save_strategy="epoch" if eval_ds else "no",
        load_best_model_at_end=True if eval_ds else False,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)] if eval_ds else []

    if class_weight == "balanced":
        _, _, train_labels = load_pairs(train_file)
        weights = compute_class_weights(train_labels)
        trainer = WeightedTrainer(
            class_weights=weights,
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            compute_metrics=make_compute_metrics_fn(),
            callbacks=callbacks,
        )
    else:
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            compute_metrics=make_compute_metrics_fn(),
            callbacks=callbacks,
        )

    trainer.train()

    # Save final best model + tokenizer
    trainer.save_model(str(model_out / "best"))
    tokenizer.save_pretrained(str(model_out / "best"))
    print(f"\n  Best model saved to {model_out / 'best'}")

    # delete checkpoints for storage
    delete_checkpoint_folders(model_out)

# LLM Augmentation Training (llm_aug_cw on WDC, llm_aug on DBLP)
WDC-Product Training

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="llm_aug_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_llm.txt"))

In [ ]:
# delete_checkpoint_folders("/content/experiments/models/llm_aug_cw_wdc-products")

DBLP-Scholar Training

In [ ]:
# train(dataset="dblp-scholar", run_name="llm_aug",
#       train_file=Path("/content/data/processed/dblp-scholar/train_aug_llm.txt"))

In [ ]:
# delete_checkpoint_folders("/content/experiments/models/llm_aug_dblp-scholar")

# Evaluation

In [ ]:
def evaluate(dataset: str, split: str = "test",
             run_name: str = "baseline",
             save_preds: bool = False):

    model_dir = MODELS / f"{run_name}_{dataset}" / "best"
    data_file = PROCESSED / dataset / f"{split}.txt"

    if not model_dir.exists():
        sys.exit(
            f"[error] Model not found at {model_dir}. "
            "Run src/baseline/train_baseline.py first."
        )
    if not data_file.exists():
        sys.exit(
            f"[error] {data_file} not found. "
            "Run src/data_prep/preprocess.py first."
        )

    print(f"\n=== Evaluating {run_name} on {dataset} [{split}] ===")


    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))

    ds = build_hf_dataset(data_file, tokenizer)

    eval_args = TrainingArguments(
        output_dir=str(MODELS / "tmp_eval"),
        per_device_eval_batch_size=64,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=eval_args)

    output = trainer.predict(ds)
    logits = output.predictions
    preds = np.argmax(logits, axis=-1).tolist()

    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    scores = (exp / exp.sum(axis=-1, keepdims=True))[:, 1].tolist()

    lefts, rights, labels = load_pairs(data_file)
    metrics = compute_metrics(labels, preds)
    cm = confusion(labels, preds)

    print(f"\n  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"\n{full_report(labels, preds)}")

    result = {
        "run": run_name,
        "dataset": dataset,
        "split": split,
        "model": str(model_dir),
        "metrics": metrics,
        "confusion_matrix": cm,
        "n_pairs": len(labels),
        "n_matches": sum(labels),
        "n_non_matches": len(labels) - sum(labels),
    }
    out_file = RESULTS / f"{run_name}_{dataset}_{split}.json"
    with open(out_file, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\n  Results saved to {out_file}")

    if save_preds:
        preds_file = RESULTS / f"{run_name}_{dataset}_{split}_preds.jsonl"
        with open(preds_file, "w") as f:
            for left, right, true_label, pred_label, score in zip(lefts, rights, labels, preds, scores):
                f.write(json.dumps({
                    "left": left,
                    "right": right,
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "score": round(score, 6),
                }) + "\n")
        print(f"  Predictions saved to {preds_file}")

    return metrics

Evaluate LLM aug

In [ ]:
# llm_wdc_metrics = evaluate(dataset="wdc-products", split="test", run_name="llm_aug_cw", save_preds=True)

In [ ]:
# llm_dblp_metrics = evaluate(dataset="dblp-scholar", split="test", run_name="llm_aug", save_preds=True)

# Web Augmentation Training (web_aug_cw on WDC, web_aug on DBLP)



WDC-Product Training

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="web_aug_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_web.txt"))

In [ ]:
# delete_checkpoint_folders("/content/experiments/models/web_aug_cw_wdc-products")

DBLP-Scholar Training

In [ ]:
# train(dataset="dblp-scholar", run_name="web_aug",
#       train_file=Path("/content/data/processed/dblp-scholar/train_aug_web.txt"))

In [ ]:
# delete_checkpoint_folders("/content/experiments/models/web_aug_dblp-scholar")

Evaluate web aug

In [ ]:
# web_wdc_metrics = evaluate(dataset="wdc-products", split="test", run_name="web_aug_cw", save_preds=True)

In [ ]:
# web_dblp_metrics = evaluate(dataset="dblp-scholar", split="test", run_name="web_aug", save_preds=True)

# WDC-v2 Training

## Balanced

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="web_v2_bal_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_web_v2_balanced.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="web_v2_bal_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_v2_balanced.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

## Hard-Negetive

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="web_v2_hn_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_web_v2_hardneg.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="web_v2_hn_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_v2_hardneg.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

# DBLP Training on Hard-Negetive

In [ ]:
# train(dataset="dblp-scholar", run_name="dblp_hardneg",
#       train_file=Path("/content/data/processed/dblp-scholar/train_aug_dblp_hardneg.txt"))

In [ ]:
# evaluate(dataset="dblp-scholar", run_name="dblp_hardneg", save_preds=True)

In [ ]:
# !zip -rq dblp_hardneg.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

# Union data training

## WDC-Products

All U

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="union_all_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_union_all.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="union_all_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_union_all.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

String U LLM

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="union_string_llm_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_union_string_llm.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="union_string_llm_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_union_string_llm.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

String U Web

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="union_string_web_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_union_string_web.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="union_string_web_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_union_string_web.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

LLM U Web

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="union_llm_web_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_union_llm_web.txt"))

In [ ]:
# evaluate(dataset="wdc-products", run_name="union_llm_web_cw", save_preds=True)

In [ ]:
# !zip -rq wdc_union_llm_web.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

## DBLP-Scholar

In [ ]:
# train(dataset="wdc-products", class_weight="balanced", run_name="union_llm_web_cw",
#       train_file=Path("/content/data/processed/wdc-products/train_aug_union_llm_web.txt"))

# Run Multiseed

In [ ]:
DEFAULT_SEEDS = [13, 42, 87]


def run_multiseed(
    dataset: str,
    run: str,
    seeds: list[int],
    class_weight: str = "none",
    train_file: str | None = None,
    epochs: int = 10,
    batch_size: int = 8,
    grad_accum: int = 4,
    lr: float = 2e-5,
):
    tf = Path(train_file) if train_file else None
    for seed in seeds:
        run_name = f"{run}_s{seed}"
        print(f"\n{'#'*70}\n# {run_name} on {dataset}  (seed={seed})\n{'#'*70}")
        train(
            dataset=dataset,
            epochs=epochs,
            batch_size=batch_size,
            grad_accum=grad_accum,
            lr=lr,
            seed=seed,
            class_weight=class_weight,
            run_name=run_name,
            train_file=tf,
        )
        delete_checkpoint_folders(MODELS / f"{run_name}_{dataset}")
        evaluate(dataset=dataset, split="test", run_name=run_name, save_preds=True)

    print(f"\nDone. Run names: {[f'{run}_s{seed}' for seed in seeds]} on {dataset}")


In [ ]:
# run_multiseed(dataset="wdc-products", run="union_all_cw",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_aug_union_all.txt")

In [ ]:
# !zip -rq wdc_seed_union_all.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

In [ ]:
# python src/baseline/run_multiseed.py --dataset dblp-scholar --run union_all --train_file data/processed/dblp-scholar/train_aug_union_all.txt


In [ ]:
# run_multiseed(dataset="dblp-scholar", run="union_all", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/dblp-scholar/train_aug_union_all.txt")

In [ ]:
# run_multiseed(dataset="wdc-products", run="sub50_cw",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_sub50.txt")

In [ ]:
# !zip -rq wdc_seed_sub50.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

In [ ]:
# run_multiseed(dataset="wdc-products", run="sub50_llm_cw",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_sub50_llm.txt")

In [ ]:
# !zip -rq wdc_seed_sub50_llm.zip /content/experiments

In [ ]:
# !zip -rq wdc_seed_sub50_llm_results.zip /content/experiments/results

In [ ]:
# from google.colab import files
# files.download('/content/wdc_seed_sub50_llm.zip')

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

In [ ]:
# run_multiseed(dataset="wdc-products", run="sub25_cw",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_sub25.txt")

In [ ]:
# !zip -rq wdc_seed_sub25.zip /content/experiments

In [ ]:
# !rm -rf /content/experiments/models/*

In [ ]:
# !rm -rf /content/experiments/results/*

In [ ]:
# run_multiseed(dataset="wdc-products", run="sub25_llm_cw",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_sub25_llm.txt")

In [ ]:
# !zip -rq wdc_seed_sub25_llm.zip /content/experiments

In [ ]:
# from google.colab import files
# files.download('/content/wdc_seed_sub25_llm.zip')

In [ ]:
# run_multiseed(dataset="dblp-scholar", run="sub50_llm", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/dblp-scholar/train_sub50_llm.txt")

In [ ]:
# !zip -rq dblp_seed_sub50_llm_results.zip /content/experiments/results

In [ ]:
# !rm -rf /content/experiments/models/*
# !rm -rf /content/experiments/results/*

In [ ]:
# run_multiseed(dataset="dblp-scholar", run="sub25", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/dblp-scholar/train_sub25.txt")

In [ ]:
# !zip -rq dblp_seed_sub25_results.zip /content/experiments/results

In [ ]:
# !rm -rf /content/experiments/models/*
# !rm -rf /content/experiments/results/*

In [ ]:
# run_multiseed(dataset="wdc-products", run="sub50_string",
#               class_weight="balanced", seeds=DEFAULT_SEEDS,
#               train_file="/content/data/processed/wdc-products/train_sub50_string.txt")

In [ ]:
# !zip -rq wdc_seed_sub50string_results.zip /content/experiments/results

In [ ]:
# !rm -rf /content/experiments/models/*
# !rm -rf /content/experiments/results/*

Multiseed training - WDC-Products

In [ ]:
run_multiseed(dataset="wdc-products", run="baseline_cw",
              class_weight="balanced", seeds=DEFAULT_SEEDS)

In [ ]:
!zip -rq wdc_seed_baseline.zip /content/experiments

In [ ]:
run_multiseed(dataset="wdc-products", run="string_aug_cw",
              class_weight="balanced", seeds=DEFAULT_SEEDS,
              train_file="/content/data/processed/wdc-products/train_aug_string.txt")

In [ ]:
!zip -rq wdc_seed_string.zip /content/experiments

In [ ]:
run_multiseed(dataset="wdc-products", run="llm_aug_cw",
              class_weight="balanced", seeds=DEFAULT_SEEDS,
              train_file="/content/data/processed/wdc-products/train_aug_llm.txt")

In [ ]:
!zip -rq wdc_seed_llm.zip /content/experiments

In [ ]:
from google.colab import files
files.download('/content/wdc_seed_llm.zip')

In [ ]:
run_multiseed(dataset="wdc-products", run="web_aug_cw",
              class_weight="balanced", seeds=DEFAULT_SEEDS,
              train_file="/content/data/processed/wdc-products/train_aug_web.txt")

In [ ]:
!zip -rq wdc_seed_web.zip /content/experiments

In [ ]:
!zip -rq wdc_seed_exp_results.zip /content/experiments

In [ ]:
from google.colab import files
files.download('/content/wdc_seed_exp_results.zip')

In [ ]:
!rm -rf /content/experiments/models/*

In [ ]:
!rm -rf /content/experiments/results/*

Multiseed training - DBLP-Scholar

In [ ]:
run_multiseed(dataset="dblp-scholar", run="baseline", seeds=DEFAULT_SEEDS)

In [ ]:
!zip -rq dblp_seed_baseline.zip /content/experiments

In [ ]:
run_multiseed(dataset="dblp-scholar", run="string_aug", seeds=[13],
              train_file="/content/data/processed/dblp-scholar/train_aug_string.txt")

In [ ]:
!zip -rq dblp_seed13_strig.zip /content/experiments

In [ ]:
run_multiseed(dataset="dblp-scholar", run="string_aug", seeds=[42],
              train_file="/content/data/processed/dblp-scholar/train_aug_string.txt")

In [ ]:
!zip -rq dblp_seed42_strig.zip /content/experiments

In [ ]:
run_multiseed(dataset="dblp-scholar", run="string_aug", seeds=[87],
              train_file="/content/data/processed/dblp-scholar/train_aug_string.txt")

In [ ]:
!zip -rq dblp_seed87_strig.zip /content/experiments

In [ ]:
# from google.colab import files
# files.download('/content/dblp_seed_strig.zip')

In [ ]:
run_multiseed(dataset="dblp-scholar", run="llm_aug", seeds=[13],
              train_file="/content/data/processed/dblp-scholar/train_aug_llm.txt")

  Precision : 0.9360
  Recall    : 0.9710
  F1        : 0.9532

In [ ]:
!zip -rq dblp_seed13_llm.zip /content/experiments

In [ ]:
run_multiseed(dataset="dblp-scholar", run="llm_aug", seeds=[42],
              train_file="/content/data/processed/dblp-scholar/train_aug_llm.txt")

In [ ]:
!zip -rq dblp_seed42_llm.zip /content/experiments

In [ ]:
run_multiseed(dataset="dblp-scholar", run="llm_aug", seeds=[87],
              train_file="/content/data/processed/dblp-scholar/train_aug_llm.txt")

In [ ]:
!zip -rq dblp_seed87_llm.zip /content/experiments

In [ ]:
# !zip -rq dblp_seed_llm.zip /content/experiments

In [ ]:
from google.colab import files
files.download('/content/dblp_seed_llm.zip')

In [ ]:
run_multiseed(dataset="dblp-scholar", run="web_aug", seeds=DEFAULT_SEEDS,
              train_file="/content/data/processed/dblp-scholar/train_aug_web.txt")

In [ ]:
!zip -rq dblp_seed_web.zip /content/experiments

In [ ]:
from google.colab import files
files.download('/content/dblp_seed_web.zip')

In [ ]:
!zip -rq dblp_seed_exp_results.zip /content/experiments